# Experiment 1 (D3a) — Decomposed vs. Single-Prompt Extraction

Tests whether splitting a form's fields across **several focused signatures** (the production pipeline) beats asking **all fields in one prompt**. Only the field grouping changes — per-field text, model, and papers are held constant.

**This notebook extracts the SINGLE arm only** and writes it as an **AI sheet** (`eval/ai sheets/staged/periodontitis/claude/<form>_single_long.csv`) — you already have the decomposed (production) results. Scoring is a separate step (see Notes); this notebook only produces the AI sheets.

**Forms:** the two *scalar* forms (`study_characteristics`, `patient_population`) are the real testbeds — they're multi-signature, so decomposed vs single is a meaningful contrast. The two *table* forms (`interventions`, `outcomes`) are level2 subforms (one row per study arm/record); they are single-signature, so decomposed == single (Δ ≈ 0) — the notebook still extracts them in the correct row-per-record shape, mainly for parity with `table_ablation` (D3b), which is their proper home.

**Kernel:** pick the env that has `dspy` (the app runs on `/home/ubuntu/miniconda3/envs/topics`). Cell 1 verifies this.

> If you edit this file on disk while it's open, use **File → Reload Notebook from Disk** to pick up the change.

In [ ]:
# Cell 1 — environment setup
import sys, os, asyncio
sys.path.insert(0, '/home/ubuntu/evistream/backend')
sys.path.insert(0, '/home/ubuntu/evistream')
os.environ.setdefault('USE_RUNTIME_BUILDERS', 'true')

print('python:', sys.executable)
try:
    import dspy; print('dspy:', dspy.__version__)
except ModuleNotFoundError:
    raise SystemExit('No dspy in this kernel — switch to the topics env kernel (it has dspy 3.0.3).')

# Load the Claude key from eval/.env
from dotenv import load_dotenv
for p in ('/home/ubuntu/evistream/eval/.env', '/home/ubuntu/evistream/backend/.env'):
    if os.path.exists(p):
        load_dotenv(p, override=False)
print('ANTHROPIC_API_KEY set:', bool(os.getenv('ANTHROPIC_API_KEY')))

In [ ]:
# Cell 2 — imports + experiment config
import pandas as pd
from eval.stage_ablation.collapse import collapse_to_single_stage
from eval.stage_ablation.run_stage_ablation import _make_config, _table_field_or_none, OUT_DIR
from eval.ablation.run_perio_ablation import PERIO_FORMS, _load_base_schema_def, _load_papers
from eval.ablation.run_extraction import _results_to_rows, _output_field_order, _write_csv, DEFAULT_CONCURRENCY
from eval.table_ablation.transform import subform_cols, explode_table_results

# ---- knobs ----
# Scalar forms: 'study_characteristics', 'patient_population' (multi-signature — real decomposed-vs-single Δ).
# Table forms (level2): 'interventions', 'outcomes' (single-signature subforms — extracted as one row per
#   study arm/record and scored; decomposed==single for these, so the Δ is ~0).
FORM            = 'study_characteristics'   # any of sorted(PERIO_FORMS)
RUN_ARMS        = ('single',)               # you already have decomposed/system results; only extract single
PAPERS_LIMIT    = 2                          # None = all 29 (full run)
CONCURRENCY     = DEFAULT_CONCURRENCY
USE_LLM         = True                       # LLM-as-judge for interpretive fields (scoring)
FORCE_REEXTRACT = False                      # True = re-extract even if the CSV already exists
print('forms available:', sorted(PERIO_FORMS))
print(f'running FORM={FORM}  RUN_ARMS={RUN_ARMS}  PAPERS_LIMIT={PAPERS_LIMIT}  FORCE_REEXTRACT={FORCE_REEXTRACT}')

## Step 1 — Inspect the collapse (free, no LLM)
Confirms the SINGLE arm merges every signature into one, preserving all fields.

In [ ]:
# Cell 3 — show decomposed vs single structure
base_def = _load_base_schema_def(FORM)
field_order = _output_field_order(base_def)
single_def = collapse_to_single_stage(base_def)

print(f"BASE {base_def['schema_name']}")
print(f"  DECOMPOSED (you already have this): {len(base_def['signatures'])} signatures, "
      f"{len(base_def.get('pipeline_stages', []))} stages")
for s in base_def['signatures']:
    print(f"    {s['class_name']:<32} fields={len(s['output_fields'])}")
print(f"  SINGLE (this notebook extracts this): 1 signature, 1 stage, "
      f"{single_def['stage_ablation']['n_fields']} fields merged")
print(f"  fields: {field_order}")

## Step 1.5 — Show the final LLM prompt (free, no LLM call)
Renders the exact message list DSPy hands to the model **after the adapter formats the signature** — system instructions, field markers (`[[ ## … ## ]]`), the prepended `reasoning` field, and the output schema. The paper markdown is replaced by a short placeholder so the prompt template is readable.

Controlled by the `FORM` knob (Cell 2) and `PROMPT_ARM` below (`'single'` = the one merged prompt; `'decomposed'` = each production signature in turn). No API key or network needed.

In [ ]:
# Cell — final prompt sent to the LLM, AFTER the DSPy adapter (markdown omitted)
from dspy.adapters.chat_adapter import ChatAdapter
try:
    from utils.caching_adapter import CachingChatAdapter
    _adapter = CachingChatAdapter()          # production adapter (prompt-cache breakpoints)
except Exception:
    _adapter = ChatAdapter()                 # plain fallback

PROMPT_ARM     = 'single'                     # 'single' = 1 merged prompt; or 'decomposed'
MD_PLACEHOLDER = '‹PAPER MARKDOWN OMITTED — full paper text is injected here at runtime›'

cfg, v_def = _make_config(FORM, base_def, PROMPT_ARM)
pipeline   = cfg.build_pipeline()

def _sig_views(extractor):
    """(label, extended-signature) pairs — handles single-call and two-stage extractors."""
    if getattr(extractor, '_is_two_stage', False):
        return [('stage1', extractor.stage1.predict.signature),
                ('stage2_row', extractor.stage2_row.predict.signature)]
    cot = extractor.extract
    return [('extract', getattr(cot, 'predict', cot).signature)]

for sig_def in v_def['signatures']:
    sig_name  = sig_def['class_name']
    extractor = pipeline._create_extractor(sig_name)
    for label, signature in _sig_views(extractor):
        inputs = {f: (MD_PLACEHOLDER if f == 'markdown_content' else f'‹{f} value›')
                  for f in signature.input_fields}
        messages = _adapter.format(signature, demos=[], inputs=inputs)

        print('═' * 100)
        print(f'ARM={PROMPT_ARM}  SIGNATURE={sig_name}  [{label}]  '
              f'({len(signature.input_fields)} inputs, {len(signature.output_fields)} outputs)')
        print('═' * 100)
        for i, msg in enumerate(messages):
            print(f'\n──── message[{i}]  role={msg["role"]} ' + '─' * 60)
            content = msg['content']
            if isinstance(content, str):
                print(content)
            else:                              # list of blocks (CachingChatAdapter cache_control)
                for b in content:
                    if b.get('cache_control'):
                        print(f'  «cache_control: {b["cache_control"]}»')
                    print(b.get('text', ''))
        print()

## Step 2 — Extract the SINGLE arm (LLM cost; cached)
Runs the real runtime path (`build_pipeline().run_batch`) on the cached periodontitis markdowns. **An arm whose CSV already exists is skipped** unless `FORCE_REEXTRACT=True`.

In [ ]:
# Cell 4 — extract RUN_ARMS only (await directly; skips cached arms)
papers = _load_papers(PAPERS_LIMIT)
OUT_DIR.mkdir(parents=True, exist_ok=True)
tbl_field = _table_field_or_none(base_def)   # set for interventions/outcomes (level2), else None
for arm in RUN_ARMS:
    path = OUT_DIR / f'{FORM}_{arm}_long.csv'
    if path.exists() and not FORCE_REEXTRACT:
        print(f"  arm '{arm}': cached → {path.name} (set FORCE_REEXTRACT=True to redo)")
        continue
    cfg, v_def = _make_config(FORM, base_def, arm)
    print(f"\n  arm '{arm}': signatures={len(v_def['signatures'])} "
          f"stages={len(v_def['pipeline_stages'])} — extracting {len(papers)} papers...")
    pipeline = cfg.build_pipeline()
    sem = asyncio.Semaphore(CONCURRENCY)
    results = await pipeline.run_batch(papers, sem)
    if tbl_field:                                   # level2 table form → one row per study arm/record
        cols = subform_cols(base_def, tbl_field)
        rows = explode_table_results(results, papers, tbl_field, cols)
        _write_csv(rows, cols, path)
    else:                                           # scalar form → one row per paper
        rows = _results_to_rows(results, papers, field_order)
        _write_csv(rows, field_order, path)
print('\n  CSVs in', OUT_DIR)

In [ ]:
# Cell 5 — eyeball the extracted single arm
for arm in RUN_ARMS:
    path = OUT_DIR / f'{FORM}_{arm}_long.csv'
    print(f'\n=== {arm} ===')
    display(pd.read_csv(path))

## Notes
- **Output:** this notebook only writes the **AI sheets** — `eval/ai sheets/staged/periodontitis/claude/<form>_<arm>_long.csv` (the extraction CSVs). No scoring / metrics files are produced here.
- **Single arm only:** `RUN_ARMS = ('single',)` — the decomposed arm is your existing production result. To also re-extract decomposed here, set `RUN_ARMS = ('decomposed', 'single')`.
- **Scalar vs table forms:** `study_characteristics` / `patient_population` are scalar (one row per paper). `interventions` / `outcomes` are level2 table forms — Cell 4 explodes them to one row per study arm/record (via `transform.explode_table_results`).
- **Caching:** Cell 4 skips any arm whose CSV exists; delete the CSV or set `FORCE_REEXTRACT=True` to redo.
- **Full run:** set `PAPERS_LIMIT = None` and re-run for each form in `sorted(PERIO_FORMS)` (`study_characteristics`, `patient_population`, `interventions`, `outcomes`).
- **Scoring (separate step):** to score the AI sheets against ground truth, run the scorer from the CLI — `python -m eval.stage_ablation.score_stage_ablation --form study_characteristics`.